In [0]:
df_silver_providers = spark.read.table("he_prod_incen_analytics_dbw_01.silver.dim_providers")
df_gold_geography = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_geography")

In [0]:
from pyspark.sql.functions import coalesce, lit, current_timestamp, row_number,col
from pyspark.sql.window import Window

# 1. Lookup geography_sk FROM Dim_Geography (Triple join for precision)
df_geo_lookup = df_gold_geography.select("geography_sk", "state", "city", "pincode").distinct()

df_gold = df_silver_providers.join(
    df_geo_lookup,
    on=["state", "city", "pincode"],
    how="left"
)

# 2. Orphan Handling for Geography
df_gold = df_gold.withColumn("geography_sk", coalesce(col("geography_sk"), lit(-1)))

# 3. Generate Surrogate Key: AUTO_INCREMENT ordered by provider_id
window_spec = Window.orderBy("provider_id")
df_gold = df_gold.withColumn("provider_sk", row_number().over(window_spec))

# 4. Add SCD Type 2 Attributes
df_gold = df_gold.withColumn("_valid_from", current_timestamp()) \
                 .withColumn("_valid_to", lit(None).cast("timestamp")) \
                 .withColumn("_is_current", lit(True).cast("boolean"))

# 5. Select ONLY the columns required in Gold (Column Pruning)
# We drop raw PII/detailed address columns, dropping replaced snowflake cols (state, city, pincode)
df_gold = df_gold.select(
    "provider_sk",
    "provider_id",                # Business Key
    "registration_number",        # Business Key for Dedup
    "provider_name",
    "provider_type",
    "specialty",
    "qualification",
    "license_number",
    "license_status",
    "board_certified",
    "years_of_experience",
    "network_status",
    "capitation_flag",
    "quality_score",
    "patient_satisfaction",
    "average_consultation_fee",
    "accepting_new_patients",
    "languages_spoken",
    "telehealth_enabled",
    "emr_system",
    "hipaa_compliant",
    "geography_sk",               # Replaces state, city, pincode
    "_valid_from",
    "_valid_to",
    "_is_current"
)

display(df_gold.limit(5))

In [0]:
gold_table_fqn = "he_prod_incen_analytics_dbw_01.gold.dim_providers"

df_gold.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(gold_table_fqn)

print(f"✅ Successfully wrote Dim_Providers to Gold layer.")